In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from statsmodels.tsa.stattools import adfuller
from src.research_config import ResearchConfig
from src.market_math import compute_log_spread


# 10 Frozen Equilibrium Shift Diagnostics
Compare frozen formation equilibrium parameters with trailing realized spreads.


In [ ]:
cfg = ResearchConfig()


In [ ]:
train = pd.read_parquet("train_prices.parquet")
test = pd.read_parquet("test_prices.parquet")
full = pd.concat([train, test])
params = pd.read_parquet("pair_parameters.parquet")
rows = []
rolling_spreads = {}

for row in params.itertuples():
    spread = compute_log_spread(full, row.dependent, row.independent, row.alpha, row.beta)
    shift = (spread.rolling(63).mean() - row.mu) / np.sqrt(row.variance)
    rolling_spreads[row.pair] = shift.reindex(test.index)
    recent = spread.iloc[-252:]
    rows.append({
        "pair": row.pair,
        "mean_absolute_shift_z": shift.reindex(test.index).abs().mean(),
        "final_shift_z": shift.iloc[-1],
        "trailing_ordinary_adf_p": float(adfuller(recent, regression="c", autolag="AIC")[1]),
    })

metrics = pd.DataFrame(rows).sort_values("mean_absolute_shift_z", ascending=False)
metrics.to_parquet("equilibrium_shift_metrics.parquet")
pd.DataFrame(rolling_spreads).to_parquet("rolling_equilibrium_shift.parquet")
display(metrics.head(15))

examples = metrics.pair.head(4).tolist()
pd.DataFrame(rolling_spreads)[examples].plot(
    figsize=(11, 4), title="Trailing 63-session mean displacement in frozen standard deviations"
)
plt.axhline(0, color="black", linestyle="--")
plt.show()
